# Titanic: Machine Learning from Disaster

**Workflow:**
1.  **Load Data**
2.  **Concatenate Datasets**
3.  **Exploratory Data Analysis (EDA)**
    *   Univariate Analysis
    *   Bivariate Analysis
    *   Multivariate Analysis
4.  **Data Preprocessing**
5.  **Modeling (Logistic Regression)**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Sklearn modules
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Visualization style
sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries Imported Successfully!")

## 2. Load Data

### Configure Data Path

In [ ]:
# USER: Update this path to where your data files are located
DATA_PATH = ""  # e.g., "C:/Users/Name/Downloads/titanic/"

import os
train_path = os.path.join(DATA_PATH, "/content/train.csv")
test_path = os.path.join(DATA_PATH, "/content/test.csv")

print(f"Data Path set to: {os.path.abspath(DATA_PATH)}")

### Read CSV Files

In [ ]:
try:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print("Datasets loaded successfully.")
except FileNotFoundError:
    print(f"Error: Files not found in {DATA_PATH}. Please check the path.")

## 3. Concatenate Datasets
We combine Train and Test datasets into one DataFrame `df` for analysis.

In [ ]:
train_len = len(train_df)
passenger_ids = test_df['PassengerId']

# Combine (drop Survived from train for now so columns match)
df = pd.concat([train_df.drop('Survived', axis=1), test_df], axis=0).reset_index(drop=True)

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")
print(f"Combined Shape: {df.shape}")

## 4. Exploratory Data Analysis (EDA)
We will analyze the data in three stages: Univariate, Bivariate, and Multivariate.

### Basic Info & Stats

In [ ]:
df.info()

In [ ]:
df.describe()

### 4.1 Univariate Analysis
Analyzing one variable at a time (Distributions, Counts, Outliers).

#### Target Variable (Survived)

In [ ]:
# Pie Chart (Train only)
plt.figure(figsize=(6, 6))
train_df['Survived'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#ff9999','#66b3ff'], explode=[0.05, 0], shadow=True)
plt.title('Survival Percentage (0=No, 1=Yes)')
plt.ylabel('')
plt.show()

#### Age Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Age'].dropna(), kde=True, color='skyblue')
plt.title('Age Distribution')
plt.show()

#### Fare Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Fare'].dropna(), kde=True, color='salmon')
plt.title('Fare Distribution')
plt.show()

#### Categorical Counts: Sex

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Sex', data=df)
plt.title('Gender Count')
plt.show()

#### Categorical Counts: Embarked

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Embarked', data=df)
plt.title('Embarked Count')
plt.show()

#### Categorical Counts: Pclass

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Pclass', data=df)
plt.title('Passenger Class Count')
plt.show()

#### Outliers: Age

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=df['Age'], color='skyblue')
plt.title('Age Boxplot')
plt.show()

#### Outliers: Fare

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=df['Fare'], color='salmon')
plt.title('Fare Boxplot')
plt.show()

### 4.2 Bivariate Analysis
Analyzing relationships between two variables (Variables vs Target).

#### Survival by Gender

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Sex', hue='Survived', data=train_df)
plt.title('Survival by Gender')
plt.show()

#### Survival by Pclass

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Pclass', hue='Survived', data=train_df)
plt.title('Survival by Passenger Class')
plt.show()

#### Age vs Survived

In [ ]:
# Violin Plot
plt.figure(figsize=(10, 6))
sns.violinplot(x='Survived', y='Age', data=train_df, palette='muted', split=True)
plt.title('Age Distribution by Survival')
plt.show()

#### Fare vs Survived

In [ ]:
# Box Plot
plt.figure(figsize=(10, 6))
sns.boxplot(x='Survived', y='Fare', data=train_df)
plt.ylim(0, 300) # Limiting y-axis to see distribution better
plt.title('Fare Distribution by Survival')
plt.show()

### 4.3 Multivariate Analysis
Analyzing interactions between multiple variables.

#### Pairplot

In [ ]:
cols_to_plot = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch', 'Survived']
sns.pairplot(train_df[cols_to_plot].dropna(), hue='Survived', palette='husl')
plt.show()

#### Correlation Heatmap

In [ ]:
numeric_df = train_df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## 5. Data Preprocessing

### Impute Missing Values

In [ ]:
imputer_age = SimpleImputer(strategy='median')
df['Age'] = imputer_age.fit_transform(df[['Age']])

df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print("Missing values imputed.")

### Feature Engineering (Titles)

In [ ]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

print(df['Title'].value_counts())

### Drop Unnecessary Columns

In [ ]:
df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1, inplace=True)
print("Unnecessary columns dropped.")

### Encode Categorical Variables

In [ ]:
label_enc = LabelEncoder()
for col in ['Sex', 'Embarked', 'Title']:
    df[col] = label_enc.fit_transform(df[col].astype(str))

print("Categorical encodings applied.")
df.head()

## 6. Modeling

### Split & Scale Data

In [ ]:
X = df.iloc[:train_len]
X_test_final = df.iloc[train_len:]
y = train_df['Survived']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test_final)

print("Data split and scaled.")

### PCA (Principal Component Analysis)

In [ ]:
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA Components retained: {X_pca.shape[1]}")

### Train Model (Logistic Regression)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_pca, y, test_size=0.2, random_state=42)
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print("Model trained.")

### Evaluate Model

In [ ]:
y_pred = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(classification_report(y_val, y_pred))

### Generate Submission

In [ ]:
final_predictions = model.predict(X_test_pca)
submission = pd.DataFrame({"PassengerId": passenger_ids, "Survived": final_predictions})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")